# CIS Lens Validation: FIONA vs GLoW

This notebook validates FIONA's axisymmetric solver (`FresnelNUFHTBatched`) against
GLoW for the **Cored Isothermal Sphere (CIS)** lens.

Two tests are performed:

1. **1-D comparison** — overplot $|F(\omega)|$ from FIONA and GLoW at $y = 0.25$ on a
   frequency grid of 560 points from $\omega = 0.1$ to $\omega = 100$, with timing.
2. **2-D contour plot** — amplitude $|F(\omega,\,y)|$ across the $\omega$–$y$ plane.

## 1. Environment Setup

In [ ]:
%matplotlib inline

import os

GL2D_DIR = (
    os.environ.get("FIONA_GL2D_DIR")
    or "/n/netscratch/dvorkin_lab/Lab/nephremidze/2-LISA/0-parallel/fift_gl2d"
)
os.environ["FIONA_GL2D_DIR"] = GL2D_DIR
os.environ.setdefault("FIONA_GL2D_STRICT", "0")

print("FIONA_GL2D_DIR =", os.environ["FIONA_GL2D_DIR"])

## 2. Imports

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

from fiona import CIS, set_num_threads, FresnelNUFHTBatched
from fiona.utils import align_global_phase

from glow import lenses as glow_lenses
from glow import time_domain_c, freq_domain_c

set_num_threads(112)

## 3. Default Parameters

In [ ]:
def default_fresnel_nufht_kwargs():
    """Return a dictionary of default FresnelNUFHTBatched constructor arguments."""
    return dict(
        gl_nodes_per_dim=128,
        min_physical_radius=1.0,
        adaptive_n_gl=True,
        auto_R_from_gl_nodes=True,
        tol=1e-9,
        window_potential=True,
        window_radius_fraction=0.75,
        window_u=True,
        window_u_width=0.02,
        use_tail_correction=True,
        n_workers=112,
        print_timing=False,
    )

## 4. CIS Validation — FIONA vs GLoW 1-D Comparison

Evaluate $F(\omega)$ for a **single CIS** lens at the origin using FIONA's axisymmetric
Hankel-transform solver and GLoW, then overplot $|F(\omega)|$ and compare timings.

**Lens**: Cored Isothermal Sphere with $\psi_0 = 1$, core radius $x_c = 0.1$.

FIONA parameterisation: `CIS(psi0, xc)`  
GLoW parameterisation: `Psi_CIS({"psi0": ..., "rc": ...}, {})`

In [ ]:
def compare_cis_fiona_vs_glow(
    w_grid=np.linspace(0.1, 100.0, 560),
    psi0=1.0,
    xc=0.1,
    y_star=0.25,
    fiona_kwargs=None,
    align_phase=True,
    font_size=15,
):
    """
    Compare FIONA and GLoW amplification factors for a single axisymmetric
    CIS lens centred on the origin.

    Returns
    -------
    F_fiona, F_glow : ndarray (complex)
        Amplification factors at each frequency in w_grid.
    """
    w_grid = np.asarray(w_grid, dtype=float)

    # ---- FIONA CIS lens ----
    lens = CIS(psi0=psi0, xc=xc)

    fk = default_fresnel_nufht_kwargs()
    if fiona_kwargs:
        fk.update(fiona_kwargs)

    solver = FresnelNUFHTBatched(lens, **fk)
    t0_fiona = time.perf_counter()
    F_arr = solver(w_grid, np.array([y_star]))
    F_fiona = F_arr[:, 0]
    t_fiona = time.perf_counter() - t0_fiona

    # ---- GLoW CIS lens (note: GLoW uses 'rc' for core radius) ----
    t0_glow = time.perf_counter()
    Psi = glow_lenses.Psi_CIS({"psi0": float(psi0), "rc": float(xc)}, {})
    It = time_domain_c.It_SingleIntegral_C(Psi, y=float(y_star))
    Fw = freq_domain_c.Fw_FFT_C(It)
    F_glow = Fw(w_grid)
    t_glow = time.perf_counter() - t0_glow

    _faster = "FIONA" if t_fiona < t_glow else "GLoW"
    _speedup = max(t_fiona, t_glow) / min(t_fiona, t_glow)
    print(f"FIONA time: {t_fiona:.3f} s | GLoW time: {t_glow:.3f} s | "
          f"{_faster} is {_speedup:.2f}x faster")

    # ---- Overplot |F(w)| ----
    F_fiona_plot = (
        align_global_phase(F_fiona, F_glow)[0] if align_phase else F_fiona
    )

    plt.rcParams.update({
        "font.family": "serif",
        "mathtext.fontset": "cm",
        "font.size": font_size,
    })
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(w_grid, np.abs(F_fiona_plot), "-",  label="FIONA", color="teal")
    ax.plot(w_grid, np.abs(F_glow),       "--", label="GLoW",  color="royalblue")
    ax.set_xlabel(r"$\omega$")
    ax.set_ylabel(r"$|F(\omega)|$")
    ax.set_title(
        rf"CIS ($\psi_0={psi0},\; x_c={xc}$) @ $y={y_star}$"
    )
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

    return F_fiona, F_glow

### 4.1 Run the 1-D Comparison

560 frequency points from $\omega = 0.1$ to $\omega = 100$, source at $y = 0.25$.

In [ ]:
F_fiona_cis, F_glow_cis = compare_cis_fiona_vs_glow(
    w_grid=np.linspace(0.1, 100.0, 560),
    psi0=1.0,
    xc=0.1,
    y_star=0.25,
    fiona_kwargs=default_fresnel_nufht_kwargs(),
)

## 5. CIS Contour Plot — $|F(\omega,\,y)|$ in the $\omega$–$y$ Plane

Compute the amplification factor over a 2-D grid of $(\omega,\,y)$ values
centred on $y = 0.25$ and display it as a filled contour plot.

In [ ]:
def plot_cis_Fwy_contour(
    psi0=1.0,
    xc=0.1,
    w_min=0.1,
    w_max=100.0,
    Nw=300,
    y_min=0.01,
    y_max=2.0,
    Ny=200,
    fiona_kwargs=None,
    cmap="magma",
    font_size=15,
):
    """
    Contour plot of |F(w, y)| for a CIS lens across the w-y plane.

    The axisymmetric solver returns F with shape (Nw, Ny), which is
    directly suitable for a 2-D image.
    """
    w_grid = np.linspace(w_min, w_max, Nw)
    y_grid = np.linspace(y_min, y_max, Ny)

    lens = CIS(psi0=psi0, xc=xc)

    fk = default_fresnel_nufht_kwargs()
    if fiona_kwargs:
        fk.update(fiona_kwargs)

    solver = FresnelNUFHTBatched(lens, **fk)

    t0 = time.perf_counter()
    F = solver(w_grid, y_grid)          # shape (Nw, Ny)
    elapsed = time.perf_counter() - t0
    print(f"FIONA computed F(w,y) on {Nw}x{Ny} grid in {elapsed:.2f} s")

    Fabs = np.abs(F)                    # amplitude |F(w, y)|

    plt.rcParams.update({
        "font.family": "serif",
        "mathtext.fontset": "cm",
        "font.size": font_size,
    })

    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.pcolormesh(
        w_grid, y_grid, Fabs.T,
        shading="auto",
        cmap=cmap,
        norm=Normalize(vmin=Fabs.min(), vmax=Fabs.max()),
    )
    ax.axhline(0.25, color="white", ls="--", lw=0.8, alpha=0.7, label=r"$y=0.25$")
    ax.set_xlabel(r"$\omega$")
    ax.set_ylabel(r"$y$")
    ax.set_title(
        rf"$|F(\omega,\,y)|$ — CIS ($\psi_0={psi0},\; x_c={xc}$)"
    )
    ax.legend(loc="upper right", framealpha=0.6)
    cbar = fig.colorbar(im, ax=ax, pad=0.02)
    cbar.set_label(r"$|F(\omega,\,y)|$")
    plt.tight_layout()
    plt.show()

    return w_grid, y_grid, F

### 5.1 Run the Contour Plot

In [ ]:
w_grid_2d, y_grid_2d, F_2d = plot_cis_Fwy_contour(
    psi0=1.0,
    xc=0.1,
    w_min=0.1,
    w_max=100.0,
    Nw=300,
    y_min=0.01,
    y_max=2.0,
    Ny=200,
    fiona_kwargs=default_fresnel_nufht_kwargs(),
)